# Lab 2D: Vector Search in Python

**Time**: ~15 min  
**Environment**: Jupyter kernel in VS Code  
**Note**: Requires Cosmos DB account with vector capability enabled and Azure OpenAI resource

In this exercise you will explore semantic similarity search using Azure Cosmos DB vector capability.

The lab follows the same structure as the C# version. Run each cell in order to complete the steps.

In [ ]:
%pip install azure-cosmos azure-identity openai python-dotenv numpy --quiet

## Step 0: Initialize Connection

Set up the Cosmos client connection and Azure OpenAI embeddings client.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
EMBEDDINGS_ENDPOINT = os.environ.get("EMBEDDINGS_ENDPOINT")
EMBEDDINGS_KEY = os.environ.get("EMBEDDINGS_KEY")
EMBEDDINGS_MODEL = os.environ.get("EMBEDDINGS_MODEL", "text-embedding-3-small")
DB_NAME = "WorkshopData"
CONT_NAME = "Docs"

for var in ["COSMOS_ENDPOINT", "EMBEDDINGS_ENDPOINT", "EMBEDDINGS_KEY"]:
    if not os.environ.get(var):
        raise RuntimeError(f"{var} environment variable is required.")

print(f"Cosmos Endpoint:     {ENDPOINT}")
print(f"Embeddings Endpoint: {EMBEDDINGS_ENDPOINT}")
print(f"Database:            {DB_NAME}")
print(f"Container:           {CONT_NAME}")
print(f"Embeddings Model:    {EMBEDDINGS_MODEL}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import DefaultAzureCredential
from openai import OpenAI

# Cosmos DB client (Entra ID auth)
cred = DefaultAzureCredential()
cosmos_client = CosmosClient(url=ENDPOINT, credential=cred)
db = cosmos_client.get_database_client(DB_NAME)
container = db.get_container_client(CONT_NAME)
print(f"Connected to Cosmos DB: {ENDPOINT}/{DB_NAME}/{CONT_NAME}")

# Embeddings client: separate Azure OpenAI resource, API key auth
# (the v1 embeddings surface does not yet support Entra ID).
embeddings_client = OpenAI(
    base_url=f"{EMBEDDINGS_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=EMBEDDINGS_KEY,
)
print(f"Embeddings client initialized (deployment: {EMBEDDINGS_MODEL})")

## Step 1: Generate Embeddings (Prebuilt)

Creates 3 sample documents, generates embeddings for them using Azure OpenAI, and stores them in the `WorkshopData/Docs` container.

In [ ]:
def embed_text(text: str) -> list[float]:
    resp = embeddings_client.embeddings.create(input=text, model=EMBEDDINGS_MODEL)
    return resp.data[0].embedding


docs = [
    {"id": "d1", "title": "Introduction to Cosmos DB", "text": "Azure Cosmos DB is a globally distributed, multi-model database service.", "partitionKey": "docs"},
    {"id": "d2", "title": "Vector Search Basics", "text": "Vector search enables semantic similarity search on embeddings.", "partitionKey": "docs"},
    {"id": "d3", "title": "Full-Text Search Guide", "text": "Full-text search in Cosmos DB supports natural language queries.", "partitionKey": "docs"},
]

for doc in docs:
    text = doc["text"]
    doc["embedding"] = embed_text(text)
    try:
        container.upsert_item(body=doc)
        ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])
        print(f"  Indexed: {doc['title']}")
        print(f"  RU charged: {ru}")
    except Exception as ex:
        print(f"  Error indexing: {ex}")

print(f"\nEmbedded {len(docs)} documents")

## Step 2: Vector Search (STUDENT EXERCISE)

Embed a query and run a vector similarity search using `VectorDistance()`.

**Expected output**: Top 2 most similar documents with similarity scores.

**Hint**: The `VectorDistance(c.embedding, @emb)` function computes similarity. Pass the query embedding as a parameter.

In [ ]:
search_text = "cosmos db vector search"
print(f"Searching for: {search_text}\n")

query_embedding = embed_text(search_text)

vector_query = """
SELECT TOP 2 c.id, c.title, c.text, VectorDistance(c.embedding, @emb) AS score
FROM c
WHERE c.partitionKey = 'docs'
ORDER BY VectorDistance(c.embedding, @emb)
"""

results = list(container.query_items(
    query=vector_query,
    parameters=[{"name": "@emb", "value": query_embedding}],
    enable_cross_partition_query=True
))

vector_query_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print("Vector search results:")
for r in results:
    print(f"  Title: {r['title']}")
    print(f"  Text: {r['text']}")
    print(f"  Score: {r['score']}\n")
print(f"RU charged: {vector_query_ru}")

## Step 3: Full-Text Search (STUDENT EXERCISE)

Run a full-text search query. This requires full-text search to be enabled on the container.

**Expected output**: Top 3 documents matching 'cosmos db' with similarity scores.

**Hint**: Use `FullTextContains(field, 'search term')` in the WHERE clause.

In [ ]:
search_text = "cosmos db"
print(f"Searching for: {search_text}\n")

fts_query = f"""
SELECT TOP 3 c.id, c.title, c.text, VectorDistance(c.embedding, @emb) AS score
FROM c
WHERE c.partitionKey = 'docs' AND FullTextContains(c, '{search_text}')
ORDER BY VectorDistance(c.embedding, @emb)
"""

fts_results = list(container.query_items(
    query=fts_query,
    parameters=[{"name": "@emb", "value": query_embedding}],
    enable_cross_partition_query=True
))

print("Full-text search results:")
for r in fts_results:
    print(f"  ID: {r.get('id')}")
    print(f"  Title: {r.get('title')}")
    print(f"  Text: {r.get('text')}\n")

print("=== Lab Complete ===")
print("You have completed the vector search exercise in Python. You:")
print("- Connected to Cosmos DB and Azure OpenAI")
print("- Generated embeddings using Azure OpenAI")
print("- Stored vectorized documents in Cosmos DB")
print("- Performed vector similarity search")
print("- Performed full-text search")